# sgRNA-level visualization of selected genes

This notebook visualizes individual-sgRNA fold changes for selected genes and compares them with the corresponding gene-level summary values. It was used to examine whether sgRNAs targeting the same gene showed consistent responses across growth conditions.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Plotting functions

The first function plots one group of selected genes. Each point represents the log2 fold change measured for an individual sgRNA, while the red horizontal marker represents the gene-level summary fold change.

The second function stacks several conditions vertically using a shared gene order and axis range. The magenta markers show the gene-level summary values, and the dashed line at log2 fold change = -1 provides a common visual reference.

In [2]:
def plot_significant_genes(df, *, title=None):

    fig, ax = plt.subplots(figsize=(6, 6))
    sns.stripplot(
        data=df, 
        x="Gene", 
        y="LFC_individual", 
        jitter=False, 
        color='blue',
        alpha=0.2, 
        ax=ax
            )
   
    # Get tick positions from the plot
    xticks = ax.get_xticks()
    xticklabels = [tick.get_text() for tick in ax.get_xticklabels()]
    tick_pos_map = dict(zip(xticklabels, xticks))
    
    # Get the summary LFC
    lfc_summary = df.groupby("Gene")["LFC_summary"].first()
    for gene, lfc in lfc_summary.items():
        if gene in tick_pos_map:
            x_pos = tick_pos_map[gene]
            ax.plot(x_pos, lfc, marker='_', markersize=12, color='red')

    #ax.set_title("Stripplot with Median")
    #ax.axhspan(-1, 1, color=grey, alpha=0.5)
    ax.tick_params("x", rotation=90)
    #ax.set_xlabel("")
    #ax.set_xticks([])
    ax.set_ylabel('log2(fold change)')
    ax.set_title(f"{title}")
    return fig

In [7]:
def plot_significant_genes_vstack(dflist, conditions, *, title=None):

    gene_list = (dflist[0]['Gene'].drop_duplicates()).tolist()
    
    fig, axs = plt.subplots(len(dflist), figsize=(12,6), sharex=True, sharey=True)
    
    for i, df in enumerate(dflist):

        sns.stripplot(
            data=df, 
            x="Gene", 
            y="LFC_individual", 
            jitter=False, 
            #color='blue',
            palette='muted',
            hue='Color',
            alpha=0.2,
            legend=False,
            order=gene_list,
            ax=axs[i]
            )
 
    # Get tick positions from the plot
    xticks = axs[2].get_xticks()
    xticklabels = [tick.get_text() for tick in axs[2].get_xticklabels()]
    tick_pos_map = dict(zip(xticklabels, xticks))
    
    # Get the summary LFC
    for i, df in enumerate(dflist):
        lfc_summary = df.groupby("Gene")["LFC_summary"].first()
        print(lfc_summary)
        for gene, lfc in lfc_summary.items():
            if gene in tick_pos_map:
                x_pos = tick_pos_map[gene]
                axs[i].plot(x_pos, lfc, marker='_', markersize=12, color='magenta')

    for i, cond in enumerate(conditions):

        axs[i].set_ylabel(cond)
        axs[i].set_xmargin(0.02)

    for ax in axs:
        ax.axhline(y=-1, color='grey', linestyle='--')
    #ax.set_title("Stripplot with Median")
    #ax.axhspan(-1, 1, color=grey, alpha=0.5)
    axs[2].tick_params("x", rotation=90)
    #ax.set_xlabel("")
    #ax.set_xticks([])
    fig.tight_layout()
    
   
    return fig

## Select genes to display

The list with the selected locus tags is loaded here.

In [ ]:
li = pd.read_excel("path/to/list", skiprows=0)
mylist=list(li['Gene'])

## Load the condition-specific results

Processed 48-hour results for LB, glucose, and succinate are loaded and restricted to the curated genes. The merge used below preserves the gene order and any additional display information from the above list.

In [ ]:

conditions = ['LB', 'Glucose', 'Succinate']
# Files in results/date/ folder
cond_file_lb = "48hLBni vs 48hLB.csv"
cond_file_glc = "48hLBni vs 48hGlc.csv"
cond_file_succ = "48hLBni vs 48hSucc.csv"

condition_files = [cond_file_lb, cond_file_glc, cond_file_succ]
dflist = [pd.read_csv(cond_file, low_memory=False) for cond_file in condition_files]

df_genes_list = [li.merge(df, on='Gene', how='left') for df in dflist]



## Generate the multi-condition figure

The resulting figure places LB, glucose, and succinate in separate aligned panels. This makes variation among sgRNAs visible while allowing the gene-level response to be compared across the three media.

In [ ]:
plot_significant_genes_vstack(df_genes_list, conditions)

The different colors correspond to functional grouping of genes (see publication). 

![sgRNA plots per function](figures/functions.png)